# SQuAD v1.1 Dataset Exploration

Stanford Question Answering Dataset v1.1 (~87K examples).  
Extractive QnA, answerable-only (no unanswerable questions).  
HuggingFace: `rajpurkar/squad`

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
from pprint import pprint
import sys
from typing import Optional, List, Dict, Any, Tuple
if '..' not in sys.path: sys.path.append('..')

from datasets import load_dataset
from datasets.arrow_dataset import Dataset
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from transformers import AutoTokenizer, PreTrainedTokenizer

tkz = AutoTokenizer.from_pretrained('bert-base-uncased')
print(f'Tokenizer vocab size: {tkz.vocab_size}')

Tokenizer vocab size: 30522


In [3]:
DATA_PATH = Path('Q:/data')
QNA_DATA_PATH = DATA_PATH / 'qna'
QNA_DATA_PATH.mkdir(parents=True, exist_ok=True)

SQUAD_V1_HF_ID = 'rajpurkar/squad'
print(f'DATA_PATH: {DATA_PATH}')
print(f'SQUAD_V1_HF_ID: {SQUAD_V1_HF_ID}')

DATA_PATH: Q:\data
SQUAD_V1_HF_ID: rajpurkar/squad


## Load SQuAD v1.1

In [4]:
# Load SQuAD v1.1 from HuggingFace
ds_sq1 = load_dataset(SQUAD_V1_HF_ID, cache_dir=str(DATA_PATH), trust_remote_code=True)
print(f'Splits: {list(ds_sq1.keys())}')
for split_name, split_ds in ds_sq1.items():
    print(f'  {split_name}: {len(split_ds)} examples')
print(f'\nFeatures: {list(ds_sq1["train"].features.keys())}')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'rajpurkar/squad' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


validation-00000-of-00001.parquet:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Splits: ['train', 'validation']
  train: 87599 examples
  validation: 10570 examples

Features: ['id', 'title', 'context', 'question', 'answers']


In [ ]:
# Examine the structure of one example
ex = ds_sq1['train'][0]
print('Keys:', list(ex.keys()))
print()
for k, v in ex.items():
    if isinstance(v, str):
        print(f'{k}: {v[:200]}{"..." if len(v) > 200 else ""}')
    elif isinstance(v, dict):
        print(f'{k}: {v}')
    elif isinstance(v, list):
        print(f'{k}: {v[:5]}{"..." if len(v) > 5 else ""}')
    else:
        print(f'{k}: {v}')

## Dataset Statistics

In [ ]:
# Convert to DataFrame for easier analysis
ds_train = ds_sq1['train']
df_sq1 = ds_train.to_pandas()
n_total = len(df_sq1)
print(f'Train DataFrame: {df_sq1.shape}')
print(f'Columns: {list(df_sq1.columns)}')
print()

# Answer counts — SQuAD v1 is all answerable, often with multiple reference answers
df_sq1['ans_count'] = df_sq1['answers'].apply(lambda a: len(a['text']))
n_empty_ans = (df_sq1['ans_count'] == 0).sum()
print(f'Rows with empty answers: {n_empty_ans}/{n_total}')
print(f'Answers per question: mean={df_sq1["ans_count"].mean():.2f}, max={df_sq1["ans_count"].max()}')
print(df_sq1['ans_count'].value_counts().sort_index())
print()

# Unique contexts and questions
n_unique_ctx = df_sq1['context'].nunique()
n_unique_q = df_sq1['question'].nunique()
n_unique_title = df_sq1['title'].nunique()
print(f'Total rows: {n_total}')
print(f'Unique titles (articles): {n_unique_title}')
print(f'Unique contexts (paragraphs): {n_unique_ctx}')
print(f'Unique questions: {n_unique_q}')
print(f'Questions per context: ~{n_total / max(n_unique_ctx, 1):.1f}')

In [ ]:
# Answer char length distribution
ans_char_lens = df_sq1['answers'].apply(lambda a: len(a['text'][0]) if len(a['text']) > 0 else 0)
print('Answer char lengths:')
print(ans_char_lens.describe())

# Questions per unique context — distribution
ctx_q_counts = df_sq1.groupby('context').size()
print(f'\nQuestions per context distribution:')
print(ctx_q_counts.describe())

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(ctx_q_counts.values, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Questions per context paragraph')
axes[0].set_xlabel('Questions')

axes[1].hist(ans_char_lens.values, bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('Answer char length')
axes[1].set_xlabel('Characters')

plt.tight_layout()
plt.show()

## Inspect Examples

In [ ]:
# Display a few examples
for i in range(5):
    row = df_sq1.iloc[i]
    context = row['context']
    question = row['question']
    answers = row['answers']

    print(f'=== Example {i} ===')
    print(f'  Title:    {row["title"]}')
    print(f'  Question: {question}')
    print(f'  Answers:  {answers["text"]}')
    print(f'  Context:  {context[:300].replace(chr(10), " ")}...')

    # Show first answer span in context
    if len(answers['text']) > 0:
        ans_text = answers['text'][0]
        ans_start = answers['answer_start'][0]
        ans_end = ans_start + len(ans_text)
        ctx_off = 40
        beg = max(ans_start - ctx_off, 0)
        ctx_around = context[beg:ans_end + ctx_off]
        print(f'  Span [{ans_start}:{ans_end}]: "{ans_text}"')
        print(f'  In context: "...{ctx_around}..."')
    print()

## Context / Answer Length Distributions

In [ ]:
# Tokenize contexts, questions, answers and measure lengths
sq1_ctx_lens = []
sq1_q_lens = []
sq1_ans_lens = []

for _, row in df_sq1.iterrows():
    ctx_toks = tkz(row['context'], add_special_tokens=False).input_ids
    q_toks = tkz(row['question'], add_special_tokens=False).input_ids
    sq1_ctx_lens.append(len(ctx_toks))
    sq1_q_lens.append(len(q_toks))
    for ans in row['answers']['text']:
        ans_toks = tkz(ans, add_special_tokens=False).input_ids
        sq1_ans_lens.append(len(ans_toks))

sq1_ctx_lens = np.array(sq1_ctx_lens)
sq1_q_lens = np.array(sq1_q_lens)
sq1_ans_lens = np.array(sq1_ans_lens)

print(f'Processed {len(sq1_ctx_lens)} examples')
print('\nContext token lengths:')
print(f'  mean={sq1_ctx_lens.mean():.1f}, median={np.median(sq1_ctx_lens):.1f}, '
      f'min={sq1_ctx_lens.min()}, max={sq1_ctx_lens.max()}, std={sq1_ctx_lens.std():.1f}')
print('Question token lengths:')
print(f'  mean={sq1_q_lens.mean():.1f}, median={np.median(sq1_q_lens):.1f}, '
      f'min={sq1_q_lens.min()}, max={sq1_q_lens.max()}, std={sq1_q_lens.std():.1f}')
print('Answer token lengths:')
print(f'  mean={sq1_ans_lens.mean():.1f}, median={np.median(sq1_ans_lens):.1f}, '
      f'min={sq1_ans_lens.min()}, max={sq1_ans_lens.max()}, std={sq1_ans_lens.std():.1f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(sq1_ctx_lens, bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Context length (tokens)')
axes[0].set_xlabel('Tokens')
axes[0].axvline(128, color='red', linestyle='--', label='128')
axes[0].axvline(256, color='orange', linestyle='--', label='256')
axes[0].axvline(512, color='green', linestyle='--', label='512')
axes[0].legend()

axes[1].hist(sq1_q_lens, bins=50, color='darkorange', edgecolor='white')
axes[1].set_title('Question length (tokens)')
axes[1].set_xlabel('Tokens')

axes[2].hist(sq1_ans_lens, bins=50, color='seagreen', edgecolor='white')
axes[2].set_title('Answer length (tokens)')
axes[2].set_xlabel('Tokens')

plt.suptitle('SQuAD v1.1 Token Length Distributions (train)')
plt.tight_layout()
plt.show()

In [ ]:
# Chunk analysis
for inp_len in [128, 256, 384, 512]:
    chunk_content = inp_len - 2  # CLS + SEP
    n_chunks = np.ceil(sq1_ctx_lens / chunk_content).astype(int)
    multi = (n_chunks > 1).sum()
    print(f'inp_len={inp_len}: need >1 chunk: {multi}/{len(sq1_ctx_lens)} ({multi/len(sq1_ctx_lens):.1%}), '
          f'max chunks: {n_chunks.max()}, mean chunks: {n_chunks.mean():.2f}')

## Deduplicate Against SQuAD v2

In [ ]:
# SQuAD v2 contains all answerable questions from v1 plus unanswerable ones.
# Load SQuAD v2 IDs so we can identify which v1 questions are already covered.
ds_sq2 = load_dataset('rajpurkar/squad_v2', cache_dir=str(DATA_PATH), trust_remote_code=True)
sq2_ids = set()
for split_name in ds_sq2.keys():
    for ex in ds_sq2[split_name]:
        sq2_ids.add(ex['id'])
print(f'SQuAD v2 unique IDs: {len(sq2_ids)}')

# Check overlap
sq1_all_ids = set()
for split_name in ds_sq1.keys():
    for ex in ds_sq1[split_name]:
        sq1_all_ids.add(ex['id'])
print(f'SQuAD v1.1 unique IDs: {len(sq1_all_ids)}')

overlap = sq1_all_ids & sq2_ids
only_v1 = sq1_all_ids - sq2_ids
print(f'Overlap (v1 IDs in v2): {len(overlap)}')
print(f'Only in v1 (not in v2): {len(only_v1)}')
print(f'Overlap ratio: {len(overlap)/len(sq1_all_ids):.1%}')

## Convert to Unified QnA Format

In [ ]:
# Convert SQuAD v1.1 to unified QnA format: (context, question, answer, source)
# Exclude questions already present in SQuAD v2 to avoid duplication
rows_unified = []
n_skipped = 0
for split_name in ds_sq1.keys():
    ds_split = ds_sq1[split_name]
    for i in range(len(ds_split)):
        ex = ds_split[i]
        if ex['id'] in sq2_ids:
            n_skipped += 1
            continue
        context = ex['context']
        question = ex['question']
        answers = ex['answers']['text']
        if not answers or not context.strip() or not question.strip():
            continue
        rows_unified.append({
            'context': context,
            'question': question,
            'answer': answers[0],
            'source': 'squad_v1',
        })

df_unified_sq1 = pd.DataFrame(rows_unified)
print(f'Skipped (already in SQuAD v2): {n_skipped}')
print(f'Unified SQuAD v1.1 rows (deduplicated): {len(df_unified_sq1)}')
df_unified_sq1.head()

In [ ]:
# Save unified format
unified_fpath = QNA_DATA_PATH / 'squad_v1_unified.parquet'
df_unified_sq1.to_parquet(unified_fpath, index=False)
print(f'Saved to {unified_fpath} ({len(df_unified_sq1)} rows)')